## Air Quality Index (AQI) — Interactive Leaflet Map

This notebook queries AQI data for the Pyrenees corridor — the French regions of Occitanie and Nouvelle-Aquitaine, and the Spanish regions of Aragon, Catalonia and Navarre — from Neo4j and renders an interactive Leaflet map with circle markers sized and colored by AQI value, a heatmap layer, and popups showing city details.

### Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI=bolt://127.0.0.1:7687
export NEO4J_USERNAME=your_username_here
export NEO4J_PASSWORD=your_password_here
export NEO4J_DATABASE=your_database_name_here
```

### Install and Load Packages

In [1]:
suppressMessages(install.packages(
  c("htmlwidgets", "httr2", "leaflet", "leaflet.extras"),
  quiet = TRUE
))

cat("Install complete.\n")

Install complete.


In [2]:
suppressMessages({
  library(htmlwidgets)
  library(httr2)
  library(leaflet)
  library(leaflet.extras)
})

### Connection Settings

In [3]:
host     <- gsub("bolt://", "http://", Sys.getenv("NEO4J_URI"), fixed = TRUE)
host     <- gsub(":7687", ":7474", host, fixed = TRUE)
user     <- Sys.getenv("NEO4J_USERNAME")
password <- Sys.getenv("NEO4J_PASSWORD")
db       <- Sys.getenv("NEO4J_DATABASE")

cat("Credentials set.\n")

Credentials set.


### Query Helper Function

In [4]:
neo4j_query <- function(cypher) {
  response <- request(paste0(host, "/db/", db, "/query/v2")) |>
    req_auth_basic(user, password) |>
    req_headers(
      "Content-Type" = "application/json",
      "Accept"       = "application/json"
    ) |>
    req_body_json(list("statement" = cypher)) |>
    req_perform()

  json   <- resp_body_json(response, simplifyVector = TRUE)
  fields <- json$data$fields
  vals   <- json$data$values

  if (is.matrix(vals)) {
    values <- as.data.frame(vals)
  } else {
    values <- as.data.frame(do.call(rbind, lapply(vals, function(row) {
      as.data.frame(t(unlist(row)), stringsAsFactors = FALSE)
    })))
  }

  colnames(values) <- fields
  values <- type.convert(values, as.is = TRUE)
  values
}

### Step 1: Query Latest AQI per City from Neo4j

We fetch the most recent Reading for each City node, along with weather data for the popup.

In [5]:
aqi_data <- neo4j_query("
  MATCH (c:City)-[:HAS_READING]->(r:Reading)
  WITH c, r ORDER BY r.timestamp DESC
  WITH c, collect(r)[0] AS latest
  RETURN c.name                AS city,
         c.country             AS country,
         c.lat                 AS lat,
         c.lon                 AS lon,
         latest.aqi_us         AS aqi_us,
         latest.aqi_category   AS aqi_category,
         latest.main_pollutant AS main_pollutant,
         latest.temperature    AS temperature,
         latest.humidity       AS humidity,
         latest.wind_speed     AS wind_speed,
         latest.timestamp      AS timestamp
  ORDER BY latest.aqi_us DESC
")

cat(sprintf("Cities loaded: %d\n", nrow(aqi_data)))
cat(sprintf("AQI range: %d - %d\n", min(aqi_data$aqi_us), max(aqi_data$aqi_us)))

Cities loaded: 102
AQI range: 1 - 157


### Step 2: Define AQI Color Scale and Marker Sizes

Colors follow the standard US AQI scale. Marker radius is proportional to AQI value, making the worst cities immediately visually prominent.

In [6]:
aqi_color <- function(aqi) {
  ifelse(aqi <= 50,  "#00E400",
  ifelse(aqi <= 100, "#FFFF00",
  ifelse(aqi <= 150, "#FF7E00",
  ifelse(aqi <= 200, "#FF0000",
  ifelse(aqi <= 300, "#8F3F97",
                     "#7E0023")))))
}

# Scale marker radius: min 8, max 30
aqi_radius <- function(aqi) {
  8 + (aqi - min(aqi)) / (max(aqi) - min(aqi)) * 22
}

aqi_data$color  <- aqi_color(aqi_data$aqi_us)
aqi_data$radius <- aqi_radius(aqi_data$aqi_us)

# Popup HTML for each city
aqi_data$popup <- paste0(
  "<b>", aqi_data$city, ", ", aqi_data$country, "</b><br/>",
  "<b>AQI (US):</b> ", aqi_data$aqi_us, "<br/>",
  "<b>Category:</b> ", aqi_data$aqi_category, "<br/>",
  "<b>Main Pollutant:</b> ", aqi_data$main_pollutant, "<br/>",
  "<b>Temperature:</b> ", aqi_data$temperature, " °C<br/>",
  "<b>Humidity:</b> ", aqi_data$humidity, "%<br/>",
  "<b>Wind Speed:</b> ", aqi_data$wind_speed, " m/s<br/>",
  "<b>Recorded:</b> ", substr(aqi_data$timestamp, 1, 19), " UTC"
)

cat("Color scale and popups prepared.\n")

Color scale and popups prepared.


### Step 3: Build the Leaflet Map

We build an interactive map with two layers the user can toggle:
- **AQI Markers** — circle markers sized and colored by AQI, with popups
- **AQI Heatmap** — a continuous heatmap showing pollution intensity across the region

In [7]:
# Centered on Pyrenees
pyrenees <- c(43.0, 1.5)

m <- leaflet() |>
  addProviderTiles(
    providers$CartoDB.Positron,
    group = "Light"
  ) |>
  addProviderTiles(
    providers$CartoDB.DarkMatter,
    group = "Dark"
  ) |>
  setView(lng = pyrenees[2], lat = pyrenees[1], zoom = 6) |>

  # Heatmap layer
  addHeatmap(
    lng       = aqi_data$lon,
    lat       = aqi_data$lat,
    intensity = aqi_data$aqi_us,
    blur      = 30,
    max       = max(aqi_data$aqi_us),
    radius    = 40,
    group     = "AQI Heatmap"
  ) |>

  # Circle markers
  addCircleMarkers(
    data        = aqi_data,
    lng         = ~lon,
    lat         = ~lat,
    radius      = ~radius,
    color       = ~color,
    fillColor   = ~color,
    fillOpacity = 0.8,
    stroke      = TRUE,
    weight      = 1.5,
    opacity     = 1,
    popup       = ~popup,
    label       = ~paste0(city, ": ", aqi_us),
    group       = "AQI Markers"
  ) |>

  # Legend
  addLegend(
    position = "bottomright",
    colors   = c("#00E400", "#FFFF00", "#FF7E00", "#FF0000", "#8F3F97", "#7E0023"),
    labels   = c("Good (0-50)", "Moderate (51-100)",
                 "Unhealthy Sensitive (101-150)", "Unhealthy (151-200)",
                 "Very Unhealthy (201-300)", "Hazardous (301+)"),
    title    = "US AQI",
    opacity  = 0.9
  ) |>

  # Layer controls
  addLayersControl(
    baseGroups    = c("Light", "Dark"),
    overlayGroups = c("AQI Markers", "AQI Heatmap"),
    options       = layersControlOptions(collapsed = FALSE)
  ) |>

  # Show markers by default, hide heatmap
  hideGroup("AQI Heatmap")

cat("Map built.\n")

Map built.


### Step 4: Save and Display

We save the map as a self-contained HTML file and display it inline.

In [8]:
map_file <- "aqi_pyrenees_map.html"
saveWidget(m, map_file, selfcontained = TRUE)
cat(sprintf("Map saved: %s\n", map_file))

Map saved: aqi_pyrenees_map.html


In [9]:
suppressMessages(library(IRdisplay))
display_html(paste0('<iframe src="', map_file, '" width="100%" height="600px"></iframe>'))